# Kiểm thử truy xuất RAG

Workflow đầy đủ: lập test pháp lý/robustness/challenge → benchmark dense/FTS/sparse và fusion trên dev → chọn cấu hình → kiểm chứng routing/graph → chạy đánh giá cuối và latency chưa cache. Toàn bộ bảng kết quả, lỗi và kết luận được lưu trong output của notebook; không sinh report rời.

Bộ 50 case gốc đã được xem ở lần triển khai trước: phần test của bộ này được ghi rõ là **reproduction**, không gọi là holdout độc lập mới. Challenge mới được khai báo trước khi tune dev; nhãn đối chiếu với corpus local, không được gọi là thẩm định pháp lý độc lập.

In [1]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ["AIAGENT_ROOT"])] if "AIAGENT_ROOT" in os.environ else []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "apps/aiAgent", parent / "frappe-bench/apps/aiAgent"])
ROOT = next((p.resolve() for p in candidates if (p / "rag/data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set AIAGENT_ROOT to the absolute aiAgent folder")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from rag.data import config  # noqa: E402 - project root must be resolved first

cfg = config()
print("Workspace:", ROOT)

Workspace: /Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent


In [2]:
import subprocess
from datetime import date

import numpy as np
import pandas as pd
from IPython.display import display
from IPython.utils.io import capture_output

from rag.data import GOLD, digest, load_corpus, read
from rag.db import connect
from rag.embedding import Embedder
from rag.evaluation import evaluate, evidence_metrics, grouped_split, select_dev
from rag.retrieval import Retriever

bundle = load_corpus()
conn = connect()
embedder = Embedder(cfg)
retriever = Retriever(conn, embedder)
record_map = {r["id"]: r for r in bundle["records"]}
split = grouped_split()
print("Snapshot:", retriever.snapshot, "Profile:", retriever.profile)
print("Gold split:", {key: len(split[key]) for key in ("dev", "test")})

/Users/capkimkhanh/Documents/DUT4_1/CSHTTT_Project/frappe-bench/apps/aiAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Fetching 22 files: 100%|██████████| 22/22 [00:00<00:00, 4167.97it/s]

Snapshot: 60a8f8c6cda5d6af3116c813 Profile: 3a2ce6e6d726a0f053748717
Gold split: {'dev': 28, 'test': 22}


## 1. Test pháp lý và source qrels
Nhãn bên dưới được kiểm tra bằng wording của record nguồn và locator. Nhãn được giữ riêng khỏi query; context view có thể cung cấp nhiều source IDs. Đây là test truy xuất bằng chứng, không là kết luận HS hay xác minh hiệu lực văn bản bên ngoài.

In [3]:
legal_cases = [
    {
        "id": "L01",
        "intent": "gir",
        "query": "GIR 1: tên phần và chương có quyết định phân loại hàng hóa không?",
        "relevant_ids": ["gir:GIR_1"],
    },
    {
        "id": "L02",
        "intent": "gir",
        "query": "GIR 2(a) hàng chưa hoàn chỉnh nhưng có đặc trưng cơ bản của hàng hoàn chỉnh",
        "relevant_ids": ["gir:GIR_2A"],
    },
    {
        "id": "L03",
        "intent": "gir",
        "query": "GIR 3(b) hàng đóng bộ bán lẻ được phân loại theo đặc tính cơ bản",
        "relevant_ids": ["gir:GIR_3B"],
    },
    {
        "id": "L04",
        "intent": "gir",
        "query": "GIR 6 khi so sánh các phân nhóm phải cùng cấp độ",
        "relevant_ids": ["gir:GIR_6"],
    },
    {
        "id": "L05",
        "intent": "gir",
        "query": "GIR 5(b) bao bì có thể dùng lặp lại có phân loại cùng hàng không?",
        "relevant_ids": ["gir:GIR_5B"],
    },
    {
        "id": "L06",
        "intent": "gir",
        "query": "Bộ bánh xăng đuých thịt bò và khoai tây chiên được phân loại vào nhóm nào theo GIR?",
        "relevant_ids": ["gir:GIR_3_NOTE_037", "gir:GIR_3_NOTE_038"],
    },
    {
        "id": "L07",
        "intent": "gir",
        "query": "Bộ thực phẩm Spaghetti, pho mát và sốt cà chua thuộc ví dụ quy tắc nào?",
        "relevant_ids": ["gir:GIR_3_NOTE_039", "gir:GIR_3_NOTE_040"],
    },
    {
        "id": "L08",
        "intent": "gir",
        "query": "Bộ đồ làm đầu gồm tông đơ, lược, kéo và khăn được phân loại vào nhóm nào?",
        "relevant_ids": ["gir:GIR_3_NOTE_MERGED_06", "gir:GIR_3_NOTE_047"],
    },
    {
        "id": "L09",
        "intent": "gir",
        "query": "Quy tắc nào nói bình thép chứa khí nén có thể dùng nhiều lần?",
        "relevant_ids": ["gir:GIR_5_NOTE_017"],
    },
    {
        "id": "L10",
        "intent": "tt85",
        "query": "Một mặt hàng có bao nhiêu mã số theo Danh mục hàng hóa?",
        "relevant_ids": ["tt85:TT85-A4-1"],
    },
    {
        "id": "L11",
        "intent": "tt85",
        "query": "Hồ sơ yêu cầu phân tích gồm tài liệu kỹ thuật hoặc bản phân tích thành phần sản phẩm",
        "relevant_ids": ["tt85:TT85-A10-1.d"],
    },
    {
        "id": "L12",
        "intent": "tt85",
        "query": "Khi lấy mẫu hàng hóa để phân tích phải lấy mấy mẫu?",
        "relevant_ids": ["tt85:TT85-A11-2.b"],
    },
    {
        "id": "L13",
        "intent": "tt85",
        "query": "Mẫu gửi cơ quan kiểm định chỉ có giá trị khi còn nguyên niêm phong",
        "relevant_ids": ["tt85:TT85-A11-2.c"],
    },
    {
        "id": "L14",
        "intent": "tt85",
        "query": "Thời hạn lưu mẫu sau thông báo kết quả phân tích là bao lâu?",
        "relevant_ids": ["tt85:TT85-A11-3.b"],
    },
    {
        "id": "L15",
        "intent": "tt85",
        "query": "Hồ sơ đăng ký danh mục đúng quy định thì cơ quan hải quan xác nhận trong mấy ngày làm việc?",
        "relevant_ids": ["tt85:TT85-A9-3.a.1"],
    },
    {
        "id": "L16",
        "intent": "tt85",
        "query": "Mẫu 01 phiếu yêu cầu phân tích kiêm biên bản lấy mẫu gồm những trường gì?",
        "relevant_ids": ["tt85:TT85-FORM-01"],
    },
    {
        "id": "L17",
        "intent": "tt85",
        "query": "Mẫu 02 phiếu tiếp nhận yêu cầu phân tích hàng hóa ghi thời gian tiếp nhận và phân công thực hiện",
        "relevant_ids": ["tt85:TT85-FORM-02"],
    },
    {
        "id": "L18",
        "intent": "tt85",
        "query": "Mẫu 03 biên bản trả lại mẫu cần thông tin người nhận và giấy ủy quyền",
        "relevant_ids": ["tt85:TT85-FORM-03"],
    },
    {
        "id": "L19",
        "intent": "tt85",
        "query": "Mẫu 04 thông báo kết quả phân tích phân loại ghi mô tả đặc tính, nhóm, phân nhóm và mã số",
        "relevant_ids": ["tt85:TT85-FORM-04"],
    },
    {
        "id": "L20",
        "intent": "tt85",
        "query": "Mẫu 05 phiếu ghi số ngày văn bản chứng từ hồ sơ hải quan bàn giao tài liệu",
        "relevant_ids": ["tt85:TT85-FORM-05"],
    },
]
for case in legal_cases:
    assert all(key in record_map for key in case["relevant_ids"])
display(
    pd.DataFrame(
        [
            {
                "case": c["id"],
                "query": c["query"],
                "qrels": c["relevant_ids"],
                "source_excerpt": " | ".join(
                    record_map[k]["source_text"][:180] for k in c["relevant_ids"]
                ),
            }
            for c in legal_cases
        ]
    )
)

,case,query,qrels,source_excerpt
0,L01,GIR 1: tên phần và chương có quyết định phân l...,[gir:GIR_1],"Tên của Phần, Chương hoặc Phân chương được đưa..."
1,L02,GIR 2(a) hàng chưa hoàn chỉnh nhưng có đặc trư...,[gir:GIR_2A],Một mặt hàng được phân loại vào một nhóm hàng ...
2,L03,GIR 3(b) hàng đóng bộ bán lẻ được phân loại th...,[gir:GIR_3B],Những hàng hóa hỗn hợp bao gồm nhiều nguyên li...
3,L04,GIR 6 khi so sánh các phân nhóm phải cùng cấp độ,[gir:GIR_6],"Để đảm bảo tính pháp lý, việc phân loại hàng h..."
4,L05,GIR 5(b) bao bì có thể dùng lặp lại có phân lo...,[gir:GIR_5B],"Ngoài Qui tắc 5(a) nêu trên, bao bì đựng hàng ..."
5,L06,Bộ bánh xăng đuých thịt bò và khoai tây chiên ...,"[gir:GIR_3_NOTE_037, gir:GIR_3_NOTE_038]",(a) Bộ thực phẩm bao gồm bánh xăng đuých làm b...
6,L07,"Bộ thực phẩm Spaghetti, pho mát và sốt cà chua...","[gir:GIR_3_NOTE_039, gir:GIR_3_NOTE_040]",(b) Bộ thực phẩm dùng để nấu món Spaghetti (mỳ...
7,L08,"Bộ đồ làm đầu gồm tông đơ, lược, kéo và khăn đ...","[gir:GIR_3_NOTE_MERGED_06, gir:GIR_3_NOTE_047]",Bộ đồ làm đầu gồm: một tông đơ điện (nhóm 85.1...
8,L09,Quy tắc nào nói bình thép chứa khí nén có thể ...,[gir:GIR_5_NOTE_017],Qui tắc này qui định việc phân loại bao bì thư...
9,L10,Một mặt hàng có bao nhiêu mã số theo Danh mục ...,[tt85:TT85-A4-1],1. Một mặt hàng có một mã số duy nhất theo Dan...


## 2. Khóa challenge và robustness suite trước khi chọn config
Challenge gồm các mã chưa xuất hiện trong 50 gold cases. Nhãn được đối chiếu mô tả source và điều kiện hàng hóa. Test scope chỉ dùng strong cues; các máy sử dụng gạo/thịt không bị loại chỉ vì nhắc nguyên liệu. Thông tin thiếu được ghi trạng thái, vẫn có thể trả ứng viên để hỗ trợ hỏi thêm.

In [4]:
challenge = [
    {
        "test_id": "C01",
        "query": "Tủ lạnh gia đình kết hợp ngăn đông, mỗi ngăn có cửa riêng, dung tích 200 lít",
        "expected_hs_8": "84181031",
    },
    {
        "test_id": "C02",
        "query": "Tủ lạnh kết hợp tủ đông gia dụng cửa riêng biệt, dung tích 300 lít",
        "expected_hs_8": "84181032",
    },
    {
        "test_id": "C03",
        "query": "Máy giặt gia đình tự động hoàn toàn, sức chứa 5 kg vải khô mỗi lần giặt",
        "expected_hs_8": "84501110",
    },
    {
        "test_id": "C04",
        "query": "Máy giặt gia đình tự động hoàn toàn giặt được 8 kg quần áo khô mỗi mẻ",
        "expected_hs_8": "84501190",
    },
    {
        "test_id": "C05",
        "query": "Máy xay trộn thức ăn gia dụng có động cơ điện gắn liền",
        "expected_hs_8": "85094000",
    },
    {
        "test_id": "C06",
        "query": "Máy ép nước rau quả dùng trong gia đình, có mô tơ điện tích hợp",
        "expected_hs_8": "85094000",
    },
    {
        "test_id": "C07",
        "query": "Máy đánh bóng sàn nhà dùng trong gia đình với động cơ điện gắn liền",
        "expected_hs_8": "85098010",
    },
    {
        "test_id": "C08",
        "query": "Thiết bị tiêu hủy rác thực phẩm trong nhà bếp, có mô tơ điện gắn liền",
        "expected_hs_8": "85098020",
    },
    {
        "test_id": "C09",
        "query": "Máy cạo râu điện có động cơ gắn liền dùng chăm sóc cá nhân",
        "expected_hs_8": "85101000",
    },
    {
        "test_id": "C10",
        "query": "Tông đơ cắt tóc điện tích hợp động cơ",
        "expected_hs_8": "85102000",
    },
    {
        "test_id": "C11",
        "query": "Máy hút bụi có mô tơ điện, công suất 1200 watt, thùng chứa 10 lít",
        "expected_hs_8": "85081100",
    },
    {
        "test_id": "C12",
        "query": "May hut bui dien 1000 W co tui hung bui 5 lit",
        "expected_hs_8": "85081100",
    },
]
original_codes = {r["expected_hs_8"] for r in read(GOLD)}
assert not ({c["expected_hs_8"] for c in challenge} & original_codes)
for case in challenge:
    assert "tt31:SXVI-T-" + case["expected_hs_8"] in record_map
challenge_hash = digest(challenge)
print("Frozen challenge hash:", challenge_hash)
display(pd.DataFrame(challenge))
robustness = [
    ("Gạo trắng đóng bao dùng làm thực phẩm", "OUT_OF_SCOPE", "product"),
    ("Thịt bò tươi để ăn", "OUT_OF_SCOPE", "product"),
    ("Áo thun bằng cotton", "OUT_OF_SCOPE", "product"),
    ("Lốp xe ô tô bằng cao su", "OUT_OF_SCOPE", "product"),
    ("Cam tươi", "OUT_OF_SCOPE", "product"),
    ("Máy xay thịt có mô tơ điện", "UNCERTAIN", "product"),
    ("Máy chế biến gạo", "UNCERTAIN", "product"),
    ("Điện thoại thông minh 5G", "UNCERTAIN", "product"),
    ("Thiết bị điện tử", "NEED_MORE_INFORMATION", "product"),
    ("Màn hình", "NEED_MORE_INFORMATION", "product"),
    ("8517.13.00", "IN_SCOPE", "lookup"),
    ("0901.21.00", "OUT_OF_SCOPE", "lookup"),
    ("123", "INVALID_CODE", "lookup"),
    ("GIR 6 phân loại phân nhóm", "LEGAL_QUERY", "gir"),
    ("Hồ sơ yêu cầu phân tích hàng hóa", "LEGAL_QUERY", "tt85"),
    ("Biên bản trả lại mẫu hàng hóa", "LEGAL_QUERY", "tt85"),
]
display(pd.DataFrame(robustness, columns=["query", "expected_status", "expected_intent"]))

Frozen challenge hash: bd13c9cbfc8ca7264508b325427dd12839ec90d6b6175a21269ad38b944d9255


,test_id,query,expected_hs_8
0,C01,"Tủ lạnh gia đình kết hợp ngăn đông, mỗi ngăn c...",84181031
1,C02,Tủ lạnh kết hợp tủ đông gia dụng cửa riêng biệ...,84181032
2,C03,"Máy giặt gia đình tự động hoàn toàn, sức chứa ...",84501110
3,C04,Máy giặt gia đình tự động hoàn toàn giặt được ...,84501190
4,C05,Máy xay trộn thức ăn gia dụng có động cơ điện ...,85094000
5,C06,"Máy ép nước rau quả dùng trong gia đình, có mô...",85094000
6,C07,Máy đánh bóng sàn nhà dùng trong gia đình với ...,85098010
7,C08,"Thiết bị tiêu hủy rác thực phẩm trong nhà bếp,...",85098020
8,C09,Máy cạo râu điện có động cơ gắn liền dùng chăm...,85101000
9,C10,Tông đơ cắt tóc điện tích hợp động cơ,85102000


,query,expected_status,expected_intent
0,Gạo trắng đóng bao dùng làm thực phẩm,OUT_OF_SCOPE,product
1,Thịt bò tươi để ăn,OUT_OF_SCOPE,product
2,Áo thun bằng cotton,OUT_OF_SCOPE,product
3,Lốp xe ô tô bằng cao su,OUT_OF_SCOPE,product
4,Cam tươi,OUT_OF_SCOPE,product
5,Máy xay thịt có mô tơ điện,UNCERTAIN,product
6,Máy chế biến gạo,UNCERTAIN,product
7,Điện thoại thông minh 5G,UNCERTAIN,product
8,Thiết bị điện tử,NEED_MORE_INFORMATION,product
9,Màn hình,NEED_MORE_INFORMATION,product


## 3. Benchmark và chọn fusion trên dev
Grid dưới đây được chốt trước final evaluation. Thử FTS, dense, BGE sparse, dense+sparse và weighted hybrid. Chọn theo name+spec Hit@10, rồi MRR@10. Query cache được dùng để so sánh ranking nhanh; latency chưa-cache được đo riêng ở cuối.

In [5]:
grid = [
    ("fts", {"fts": 1}),
    ("dense", {"dense": 1}),
    ("sparse", {"sparse": 1}),
    ("hybrid", {"dense": 1, "fts": 1}),
    ("hybrid", {"dense": 1, "fts": 0.25}),
    ("dense_sparse", {"dense": 1, "sparse": 0.5}),
    ("dense_sparse", {"dense": 1, "sparse": 1}),
    ("dense_sparse", {"dense": 1, "sparse": 2}),
    ("hybrid_sparse", {"dense": 1, "sparse": 1, "fts": 0.1}),
]
dev_runs = []
with capture_output() as dev_progress:
    for mode, weights in grid:
        dev_runs.append(evaluate(retriever, "dev", mode, weights=weights))
dev_table = pd.DataFrame(
    [
        {"mode": r["config"]["mode"], "weights": r["config"]["weights"], "query_variant": v, **m}
        for r in dev_runs
        for v, m in r["metrics"].items()
    ]
)
display(
    dev_table[["mode", "weights", "query_variant", "count", "hit@1", "hit@5", "hit@10", "mrr@10"]]
)
selected = select_dev(dev_runs)
retriever = Retriever(conn, embedder)
print(
    "Selected on DEV only:",
    {k: selected[k] for k in ("selected_mode", "weights", "selection_metric")},
)

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


,mode,weights,query_variant,count,hit@1,hit@5,hit@10,mrr@10
0,fts,{'fts': 1},name_only,28,0.250000,0.321429,0.535714,0.314626
1,fts,{'fts': 1},name_and_spec,28,0.214286,0.535714,0.535714,0.333333
2,dense,{'dense': 1},name_only,28,0.607143,0.785714,0.857143,0.697917
3,dense,{'dense': 1},name_and_spec,28,0.642857,1.000000,1.000000,0.803571
4,sparse,{'sparse': 1},name_only,28,0.642857,0.714286,0.785714,0.676531
5,sparse,{'sparse': 1},name_and_spec,28,0.678571,1.000000,1.000000,0.827381
6,hybrid,"{'dense': 1, 'fts': 1}",name_only,28,0.321429,0.678571,0.785714,0.458333
7,hybrid,"{'dense': 1, 'fts': 1}",name_and_spec,28,0.464286,0.750000,0.964286,0.607185
8,hybrid,"{'dense': 1, 'fts': 0.25}",name_only,28,0.357143,0.714286,0.857143,0.512599
9,hybrid,"{'dense': 1, 'fts': 0.25}",name_and_spec,28,0.535714,0.964286,0.964286,0.698810


Selected on DEV only: {'selected_mode': 'dense_sparse', 'weights': {'dense': 1, 'sparse': 2}, 'selection_metric': 'name_and_spec Hit@10, then MRR@10'}


## 4. Kiểm chứng evidence, routing và graph
Source recall là tỷ lệ qrels được phủ bởi các matches/context source IDs trong top 5. Suite này được curator đối chiếu corpus local; chưa có expert review độc lập. Graph ưu tiên constraints, lấy đủ ancestors trước notes và giữ danh sách evidence bị bỏ khi vượt budget.

In [6]:
legal_result = evidence_metrics(retriever, legal_cases)
display(pd.DataFrame(legal_result["cases"]))
print("Mean source recall@5:", legal_result["mean_source_recall@5"])
print("Label provenance:", legal_result["annotation_status"])

,case,intent,query,expected,returned,source_recall@5,citation_complete
0,L01,gir,GIR 1: tên phần và chương có quyết định phân l...,[gir:GIR_1],"[gir:GIR_1_NOTE_002, gir:GIR_1_NOTE_001, gir:G...",1.0,True
1,L02,gir,GIR 2(a) hàng chưa hoàn chỉnh nhưng có đặc trư...,[gir:GIR_2A],"[gir:GIR_2A, gir:GIR_2_NOTE_002, gir:GIR_2_NOT...",1.0,True
2,L03,gir,GIR 3(b) hàng đóng bộ bán lẻ được phân loại th...,[gir:GIR_3B],"[gir:GIR_3B, gir:GIR_3_NOTE_030, gir:GIR_3_NOT...",1.0,True
3,L04,gir,GIR 6 khi so sánh các phân nhóm phải cùng cấp độ,[gir:GIR_6],"[gir:GIR_6_NOTE_003, gir:GIR_6, gir:GIR_6_NOTE...",1.0,True
4,L05,gir,GIR 5(b) bao bì có thể dùng lặp lại có phân lo...,[gir:GIR_5B],"[gir:GIR_5_NOTE_017, gir:GIR_5B, gir:GIR_5_NOT...",1.0,True
5,L06,gir,Bộ bánh xăng đuých thịt bò và khoai tây chiên ...,"[gir:GIR_3_NOTE_037, gir:GIR_3_NOTE_038]","[gir:GIR_3_NOTE_038, gir:GIR_3_NOTE_037, gir:G...",1.0,True
6,L07,gir,"Bộ thực phẩm Spaghetti, pho mát và sốt cà chua...","[gir:GIR_3_NOTE_039, gir:GIR_3_NOTE_040]","[gir:GIR_3_NOTE_040, gir:GIR_3_NOTE_039, gir:G...",1.0,True
7,L08,gir,"Bộ đồ làm đầu gồm tông đơ, lược, kéo và khăn đ...","[gir:GIR_3_NOTE_047, gir:GIR_3_NOTE_MERGED_06]","[gir:GIR_3_NOTE_047, gir:GIR_3_NOTE_MERGED_06,...",1.0,True
8,L09,gir,Quy tắc nào nói bình thép chứa khí nén có thể ...,[gir:GIR_5_NOTE_017],"[gir:GIR_5_NOTE_017, gir:GIR_5_NOTE_016, gir:G...",1.0,True
9,L10,tt85,Một mặt hàng có bao nhiêu mã số theo Danh mục ...,[tt85:TT85-A4-1],"[tt85:TT85-A4-1, tt85:TT85-A2-3, tt85:TT85-A13...",1.0,True


Mean source recall@5: 1.0
Label provenance: CURATED_AGAINST_LOCAL_SOURCE_NOT_INDEPENDENT_EXPERT_REVIEW


In [7]:
routing_rows = []
for query, status, intent in robustness:
    routing = retriever.route(query)
    routing_rows.append(
        {
            "query": query,
            "expected": status,
            "actual": routing["status"],
            "intent": routing["intent"],
            "passed": routing["status"] == status and routing["intent"] == intent,
        }
    )
display(pd.DataFrame(routing_rows))
assert all(r["passed"] for r in routing_rows)
outside = retriever.routed_search("Áo thun bằng cotton")
assert not outside["candidates"]
unknown = retriever.lookup("99999999")
assert unknown["status"] == "CODE_NOT_IN_CORPUS"
assert not retriever.search(
    "phân loại", intent="tt85", mode="fts", as_of=date(2026, 9, 14), expand=False
)["matches"]
assert retriever.search(
    "phân loại", intent="tt85", mode="fts", as_of=date(2026, 9, 15), expand=False
)["matches"]
print("Scope, missing-information, unknown-code and local validity checks: PASS")

,query,expected,actual,intent,passed
0,Gạo trắng đóng bao dùng làm thực phẩm,OUT_OF_SCOPE,OUT_OF_SCOPE,product,True
1,Thịt bò tươi để ăn,OUT_OF_SCOPE,OUT_OF_SCOPE,product,True
2,Áo thun bằng cotton,OUT_OF_SCOPE,OUT_OF_SCOPE,product,True
3,Lốp xe ô tô bằng cao su,OUT_OF_SCOPE,OUT_OF_SCOPE,product,True
4,Cam tươi,OUT_OF_SCOPE,OUT_OF_SCOPE,product,True
5,Máy xay thịt có mô tơ điện,UNCERTAIN,UNCERTAIN,product,True
6,Máy chế biến gạo,UNCERTAIN,UNCERTAIN,product,True
7,Điện thoại thông minh 5G,UNCERTAIN,UNCERTAIN,product,True
8,Thiết bị điện tử,NEED_MORE_INFORMATION,NEED_MORE_INFORMATION,product,True
9,Màn hình,NEED_MORE_INFORMATION,NEED_MORE_INFORMATION,product,True


Scope, missing-information, unknown-code and local validity checks: PASS


In [8]:
graph_query = "Điện thoại thông minh dùng mạng di động tế bào, hệ điều hành chạy nhiều ứng dụng của bên thứ ba"
graph_result = retriever.search(graph_query, expand=True)
evidence = graph_result["evidence"]
display(
    pd.DataFrame(
        [{"code": r["code"], "rank": i + 1} for i, r in enumerate(graph_result["candidates"])]
    )
)
display(
    pd.DataFrame(
        [
            {
                "id": r["id"],
                "required": r["required_by_graph"],
                "query_similarity": r["query_similarity"],
                "source": r["source_text"][:200],
                "locator": r["locator"],
            }
            for r in evidence["records"]
        ]
    )
)
print(
    "Graph coverage:",
    {
        k: evidence[k]
        for k in ("coverage_status", "required_ids", "missing_required_ids", "characters_used")
    },
)
assert "tt31:SXVI-N-CH85_NOTE_5" in {r["id"] for r in evidence["records"]}
assert len(evidence["records"]) <= cfg["evidence_limit"]
assert evidence["characters_used"] <= evidence["character_budget"]
assert all(r["locator"] for r in evidence["records"])
# Demonstrate an explicit budget failure instead of silently presenting incomplete evidence.
bounded = retriever.evidence(graph_result["candidates"], limit=1, max_chars=100)
print("Small-budget result:", bounded["coverage_status"], "omitted:", len(bounded["omitted_ids"]))
assert bounded["omitted_ids"]

constraint_example = retriever.lookup("852581")
constraint_evidence = constraint_example["evidence"]
print("Camera HS6 explicit constraint:", constraint_evidence["required_ids"])
assert "tt31:SXVI-N-CH85_SUBNOTE_1" in constraint_evidence["required_ids"]
assert not constraint_evidence["missing_required_ids"]
constrained_budget = retriever.evidence(constraint_example["matches"], limit=1, max_chars=1)
assert constrained_budget["missing_required_ids"]
print(
    "Mandatory constraint omitted by budget is reported:",
    constrained_budget["missing_required_ids"],
)

,code,rank
0,85177921,1
1,85177910,2
2,85176269,3
3,85176243,4
4,85176299,5
5,85171300,6
6,85176249,7
7,85176251,8
8,85171400,9
9,85176259,10


,id,required,query_similarity,source,locator
0,tt31:SXVI-N-CH85_NOTE_5,False,0.676151,"Theo mục đích của nhóm 85.17, thuật ngữ ""điện ...","{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
1,tt31:SXVI-C-CH85_P12_CTX_001,False,0.654452,"Bộ điện thoại, kể cả điện thoại thông minh và ...","{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
2,tt31:SXVI-N-CH85_NOTE_6B,False,0.521461,(b) Khái niệm “thẻ thông minh” nghĩa là thẻ đư...,"{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
3,tt31:SXVI-N-CH85_NOTE_12B_II,False,0.491343,(ii) Mạch tích hợp lai trong đó các phần tử th...,"{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
4,tt31:SXVI-N-CH85_NOTE_9,False,0.490809,"Theo mục đích của nhóm 85.36, “đầu nối dùng ch...","{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
5,tt31:SXVI-C-CH85_P12_CTX_002,False,0.471609,"Thiết bị khác để phát hoặc nhận tiếng, hình ản...","{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
6,tt31:SXVI-N-CH85_NOTE_12B_III,False,0.468360,(iii) Mạch tích hợp đa chíp bao gồm hai hoặc n...,"{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
7,tt31:SXVI-C-CH85_P12_CTX_008,False,0.468160,"Của thiết bị truyền dẫn, trừ loại dùng cho phá...","{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
8,tt31:SXVI-C-CH85_P12_CTX_003,False,0.459049,Thiết bị dùng cho hệ thống hữu tuyến sóng mang...,"{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."
9,tt31:SXVI-N-CH85_NOTE_12A_I,False,0.454457,(a) (i) “Thiết bị bán dẫn” là các thiết bị bán...,"{'pdf': 'TT_31-2022_Section-XVI.pdf', 'documen..."


Graph coverage: {'coverage_status': 'BUDGET_LIMITED', 'required_ids': [], 'missing_required_ids': [], 'characters_used': 10532}
Small-budget result: BUDGET_LIMITED omitted: 93
Camera HS6 explicit constraint: ['tt31:SXVI-N-CH85_SUBNOTE_1']
Mandatory constraint omitted by budget is reported: ['tt31:SXVI-N-CH85_SUBNOTE_1']


## 5. Đánh giá cuối — config đã khóa
Chạy lại 22 test cases cũ là reproduction. Challenge mới dùng query/mã chưa có trong bộ 50 và được chốt ở đầu notebook. Không điều chỉnh config từ kết quả dưới đây.

In [9]:
assert digest(challenge) == challenge_hash
with capture_output() as final_progress:
    reproduction = evaluate(
        retriever, "test", selected["selected_mode"], weights=selected["weights"]
    )
    challenge_result = evaluate(
        retriever,
        "challenge",
        selected["selected_mode"],
        weights=selected["weights"],
        cases=challenge,
    )
final_table = pd.DataFrame(
    [
        {"evaluation": r["config"]["evaluation_status"], "variant": v, **m}
        for r in [reproduction, challenge_result]
        for v, m in r["metrics"].items()
    ]
)
display(final_table)
failures = [
    {
        "suite": r["config"]["split"],
        "case": x["test_id"],
        "query_variant": x["variant"],
        "expected": x["gold"],
        "returned": x["codes"],
    }
    for r in [reproduction, challenge_result]
    for x in r["results"]
    if x["gold_rank"] is None
]
display(pd.DataFrame(failures))

,evaluation,variant,count,hit@1,hit@3,hit@5,hit@10,mrr@10,macro_code_hit@10,latency_p50_ms,latency_p95_ms,failures@10
0,PREVIOUSLY_EXPOSED_REPRODUCTION,name_only,22,0.454545,0.454545,0.500000,0.590909,0.479978,0.677778,108.906542,116.880200,"[E006, E007, E010, E011, E012, E013, E015, E02..."
1,PREVIOUSLY_EXPOSED_REPRODUCTION,name_and_spec,22,0.818182,0.863636,0.909091,0.954545,0.857323,0.944444,125.949042,132.428765,[E015]
2,SOURCE_GROUNDED_CHALLENGE,query,12,0.666667,0.833333,0.916667,1.000000,0.778571,1.000000,113.849771,120.265148,[]


,suite,case,query_variant,expected,returned
0,test,E006,name_only,84714990,"[85234914, 85234993, 84253900, 84251900, 84378..."
1,test,E007,name_only,84715010,"[85437040, 85369039, 85389020, 85389019, 85433..."
2,test,E010,name_only,84716040,"[84703000, 84701000, 84702100, 84702900, 84705..."
3,test,E011,name_only,84716040,"[85394100, 85393110, 85412100, 85414100, 85393..."
4,test,E012,name_only,84716040,"[84703000, 84702100, 84701000, 84702900, 84713..."
5,test,E013,name_only,84716040,"[85235910, 85232943, 85291021, 85291093, 85232..."
6,test,E015,name_only,84716090,"[85291030, 84251900, 84253900, 84251100, 84253..."
7,test,E015,name_and_spec,84716090,"[85272110, 85176210, 85176253, 85176292, 85176..."
8,test,E025,name_only,85171300,"[84818021, 85232963, 85232943, 84818022, 85232..."
9,test,E033,name_only,85044019,"[85444929, 85444949, 85444919, 85444939, 85444..."


## 6. Latency chưa cache và kiểm chứng mã
Model đã load, nhưng từng query được encode thật với `use_cache=False`. Tách chi phí embedding, dense, sparse, FTS và graph; mẫu đo nhỏ chỉ là local measurement.

In [10]:
latency_queries = [c["query"] for c in challenge[:6]]
latency_rows = []
with capture_output() as latency_progress:
    for query in latency_queries:
        result = retriever.search(query, use_cache=False, expand=True)
        latency_rows.append({"query": query, **result["timings"]})
display(pd.DataFrame(latency_rows))
assert not any(r["query_cache_hit"] for r in latency_rows)
print(
    "Uncached-query p50/p95 ms:",
    np.percentile([r["total_ms"] for r in latency_rows], [50, 95]).round(1),
)

,query,embedding_ms,query_cache_hit,dense_ms,sparse_ms,expansion_ms,total_ms
0,"Tủ lạnh gia đình kết hợp ngăn đông, mỗi ngăn c...",69.443417,False,14.860125,5.488416,12.805000,102.717167
1,Tủ lạnh kết hợp tủ đông gia dụng cửa riêng biệ...,85.210792,False,23.269667,6.078542,10.229625,124.857666
2,"Máy giặt gia đình tự động hoàn toàn, sức chứa ...",84.753833,False,20.386833,6.269916,10.099375,121.576000
3,Máy giặt gia đình tự động hoàn toàn giặt được ...,85.748709,False,13.854792,7.666458,8.944292,116.277542
4,Máy xay trộn thức ăn gia dụng có động cơ điện ...,87.139791,False,23.535792,9.194417,12.777792,132.714042
5,"Máy ép nước rau quả dùng trong gia đình, có mô...",87.416584,False,17.150292,8.794500,11.074750,124.508041


Uncached-query p50/p95 ms: [123.  130.7]


In [11]:
import os

env = {**os.environ, "RAG_RUN_DB_TESTS": "1", "TOKENIZERS_PARALLELISM": "false"}
checks = [
    [str(ROOT / ".venv/bin/ruff"), "check", "rag", "tests", "notebooks"],
    [str(ROOT / ".venv/bin/mypy"), "rag"],
    [str(ROOT / ".venv/bin/bandit"), "-r", "rag", "-ll", "-q"],
    [str(ROOT / ".venv/bin/python"), "-m", "pytest", "tests", "-q"],
]
for command in checks:
    completed = subprocess.run(
        command, cwd=ROOT, env=env, capture_output=True, text=True, check=False
    )
    print(" ".join(command[1:]), "=>", completed.returncode)
    print(completed.stdout or completed.stderr or "PASS")
    assert completed.returncode == 0
conn.close()

check rag tests notebooks => 0
All checks passed!



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


rag => 0
Success: no issues found in 6 source files



-r rag -ll -q => 0
PASS


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


-m pytest tests -q => 0
....................                                                     [100%]
20 passed in 5.77s



## Kết luận
Pipeline đã bao gồm GIR contextual views, TT85 clauses/subpoints và 5 forms, dense+sparse embeddings, weighted hybrid, intent/scope states, bounded graph evidence, tests và benchmark. Xem các bảng trên để xác định chất lượng thực tế và các case thất bại. Output được giữ trong chính notebook.

Các nhãn pháp lý/challenge được đối chiếu corpus local, chưa được chuyên gia độc lập xác nhận. Scope heuristic trên suite nhỏ chưa thay thế bộ scope benchmark diện rộng. Hiệu lực TT85 đang dựa metadata local. LLM generation, final HS classification, GIR reasoning engine, Critic và ERP integration thuộc giai đoạn sau.